# 12 - Phase 4: calibration and uncertainty

**Main question**: when MiniConvNet (or the strongest Phase 2 baseline) reports a high-confidence
prediction, should that confidence be trusted?

**No retraining.** Every model here is either loaded from an existing `.keras` checkpoint and run
forward (a single evaluation pass, or ~10-20 stochastic passes for MC Dropout) or read from
already-saved probabilities. There is no `.fit()` call anywhere in this notebook or in
`src/calibration_utils.py`.

**No new models.** Primary is MiniConvNet; secondary is whichever Phase 2 `run_type == 'finetuned'`
baseline has the highest accuracy - **read programmatically from
`results/fair_baseline/metrics_fair_baseline.csv`, exactly as Phase 3 did**, not hardcoded. As of the
last Phase 2 run this is VGG16 (77.14%), but this notebook does not assume that - see the next cell.

**Which predictions get calibrated - verified, not assumed.** Calibrating stale predictions from an
older run would silently answer a different question than intended (Phase 3's Grad-CAM findings are
tied to these exact two checkpoint instances, so Phase 4's calibration numbers need to be tied to the
same instances to be comparable). `calibration_utils.locate_or_generate_predictions()`:

1. searches `results/fair_baseline/predictions/` and `outputs/predictions/` for a file whose name is
   an **exact** match for the run being calibrated (`miniconvnet_single_run`, `vgg16_runB_finetuned`);
2. **rejects and prints** any file for the same model under a different run name (e.g. the older,
   unfair `vgg16_faithful_predictions.csv`, or `miniconvnet_faithful_predictions.csv` /
   `miniconvnet_clean_predictions.csv` from the original CV-based runs) - these are never silently
   substituted;
3. if no exact match exists, verifies the checkpoint file itself exists (never retrains a substitute)
   and generates predictions via **one forward pass** - explicitly logged as such, not hidden.

Validation-set predictions were never saved by any earlier phase (Phase 2/3 only evaluated on test),
so they are always generated fresh here - that is expected, not a fallback from a failed search.

**CPU only.** No CUDA. MC Dropout is scoped to 10-20 passes as instructed - print a time estimate
after the first pass (informational, no stop gate, the same pattern Phase 3 used for Grad-CAM timing).

**Nothing in this notebook has been executed.** It was written, not run - checkpoints and predictions
only exist on the machine that will actually run it.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

import time
import numpy as np
import pandas as pd

from src.config import *
from src.data_utils import load_split

from src.gradcam_utils import identify_strongest_finetuned_baseline, verify_required_checkpoints
from src.finetune_utils import CHECKPOINTS_LOCAL

from src.calibration_utils import (
    ensure_calibration_dirs, CALIBRATION_DIR, CALIBRATION_FIGURES_DIR,
    MC_DROPOUT_RESULTS_JSON,
    locate_or_generate_predictions,
    calibration_summary, expected_calibration_error, brier_score, negative_log_likelihood,
    apply_temperature, fit_temperature, before_after_comparison,
    has_batchnorm, mc_dropout_predict, mc_dropout_correct_vs_incorrect,
    print_mc_dropout_time_estimate,
    plot_reliability_diagram, plot_confidence_distribution, plot_confidence_by_correctness,
    plot_uncertainty_vs_correctness,
    record_calibration_result, load_calibration_results, save_mc_dropout_summary,
)

pd.set_option('display.width', 200)
print('calibration output dirs:', ensure_calibration_dirs())

calibration output dirs: {'calibration_dir': 'C:\\Users\\shrey\\Downloads\\MIP_project_part2\\Lung-Cancer-Classification-\\Lung-Cancer-Classification-\\results\\calibration', 'figures': 'C:\\Users\\shrey\\Downloads\\MIP_project_part2\\Lung-Cancer-Classification-\\Lung-Cancer-Classification-\\results\\calibration\\figures', 'predictions_used': 'C:\\Users\\shrey\\Downloads\\MIP_project_part2\\Lung-Cancer-Classification-\\Lung-Cancer-Classification-\\results\\calibration\\predictions_used'}


## Step 0a - identify the secondary model programmatically (same logic as Phase 3)

In [2]:
secondary = identify_strongest_finetuned_baseline()

if secondary['available']:
    print('Strongest fine-tuned baseline from Phase 2 (secondary model for this phase):')
    print(f"  model      : {secondary['model']}")
    print(f"  run_name   : {secondary['run_name']}")
    print(f"  accuracy   : {secondary['accuracy']:.4f}")
    print(f"  checkpoint : {secondary['checkpoint_path']}")
else:
    print('Could not determine a secondary model:')
    print(' ', secondary['reason'])
    print('Run notebook 09 (Phase 2) first. Secondary-model sections below will be skipped;')
    print('MiniConvNet-only calibration can still proceed once its checkpoint is verified.')

PRIMARY_RUN_NAME = 'miniconvnet_single_run'
PRIMARY_CHECKPOINT = str(CHECKPOINTS_LOCAL / f'{PRIMARY_RUN_NAME}.keras')
SECONDARY_RUN_NAME = secondary['run_name'] if secondary['available'] else None
SECONDARY_MODEL_NAME = secondary['model'] if secondary['available'] else None
SECONDARY_CHECKPOINT = secondary['checkpoint_path'] if secondary['available'] else None

Strongest fine-tuned baseline from Phase 2 (secondary model for this phase):
  model      : VGG16
  run_name   : vgg16_runB_finetuned
  accuracy   : 0.7714
  checkpoint : C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\checkpoints_local\vgg16_runB_finetuned.keras


## Step 0b - verify both checkpoints exist BEFORE anything else runs

Same gate as Phase 3: if a checkpoint is missing, this notebook does **not** retrain a substitute -
it prints exactly which file is missing and the corresponding section is skipped.

In [3]:
ck_status = verify_required_checkpoints(
    PRIMARY_CHECKPOINT, SECONDARY_CHECKPOINT,
    secondary_label=f"{SECONDARY_MODEL_NAME or 'secondary'} (strongest Phase 2 baseline)")

print(ck_status['primary']['message'])
print(ck_status['secondary']['message'])

PRIMARY_OK = ck_status['primary']['exists']
SECONDARY_OK = ck_status['secondary']['exists']
print()
print('PRIMARY_OK  :', PRIMARY_OK)
print('SECONDARY_OK:', SECONDARY_OK)

OK - MiniConvNet (primary) checkpoint found at C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\checkpoints_local\miniconvnet_single_run.keras
OK - VGG16 (strongest Phase 2 baseline) checkpoint found at C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\checkpoints_local\vgg16_runB_finetuned.keras

PRIMARY_OK  : True
SECONDARY_OK: True


In [4]:
sdf = load_split('faithful')
print(sdf['split'].value_counts().to_dict())

{'train': 613, 'test': 315, 'val': 72}


## STEP 1 - locate (or generate) test AND validation probabilities for both models

For each available model: test-set predictions first (searched, then generated only if truly
absent), then validation-set predictions (always generated, per the note above). Read the printed
`USING:` line for each - that is the provenance record for every number that follows.

In [5]:
predictions = {}

if PRIMARY_OK:
    print('=== MiniConvNet ===')
    print('-- test split --')
    predictions[('MiniConvNet', 'test')] = locate_or_generate_predictions(
        PRIMARY_RUN_NAME, PRIMARY_CHECKPOINT, sdf, 'test', one_hot=True,
        model_short_name='miniconvnet')
    print('-- val split --')
    predictions[('MiniConvNet', 'val')] = locate_or_generate_predictions(
        PRIMARY_RUN_NAME, PRIMARY_CHECKPOINT, sdf, 'val', one_hot=True,
        model_short_name='miniconvnet')
else:
    print('Skipping MiniConvNet - checkpoint missing (see Step 0b).')

if SECONDARY_OK:
    print(f'=== {SECONDARY_MODEL_NAME} ===')
    print('-- test split --')
    predictions[(SECONDARY_MODEL_NAME, 'test')] = locate_or_generate_predictions(
        SECONDARY_RUN_NAME, SECONDARY_CHECKPOINT, sdf, 'test', one_hot=False,
        model_short_name=SECONDARY_MODEL_NAME.lower())
    print('-- val split --')
    predictions[(SECONDARY_MODEL_NAME, 'val')] = locate_or_generate_predictions(
        SECONDARY_RUN_NAME, SECONDARY_CHECKPOINT, sdf, 'val', one_hot=False,
        model_short_name=SECONDARY_MODEL_NAME.lower())
else:
    print(f'Skipping {SECONDARY_MODEL_NAME or "secondary model"} - checkpoint missing/undetermined.')

MODELS_AVAILABLE = sorted({m for (m, _split) in predictions})
print()
print('models with usable predictions:', MODELS_AVAILABLE)

=== MiniConvNet ===
-- test split --
  3 prediction file(s) for 'miniconvnet' were found under a DIFFERENT run name and are REJECTED (would calibrate the wrong checkpoint instance, not 'miniconvnet_single_run'):
    REJECTED: C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\outputs\predictions\lc25000_miniconvnet_predictions.csv
    REJECTED: C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\outputs\predictions\miniconvnet_clean_predictions.csv
    REJECTED: C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\outputs\predictions\miniconvnet_faithful_predictions.csv
  No file named 'miniconvnet_single_run_predictions.csv' exists in C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\fair_baseline\predictions or C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Can

## STEP 2 - calibration metrics (test set, BEFORE any calibration)

Accuracy, mean confidence, ECE, Brier score, NLL - for each model, computed on the exact test-set
probabilities located/generated in Step 1.

In [6]:
before_summaries = {}
for model_name in MODELS_AVAILABLE:
    test = predictions[(model_name, 'test')]
    summary = calibration_summary(test['y_true'], test['y_prob'])
    before_summaries[model_name] = summary
    print(f'--- {model_name} (test, before calibration) ---')
    print(f"  n_samples             : {summary['n_samples']}")
    print(f"  accuracy              : {summary['accuracy']:.4f}")
    print(f"  mean_confidence       : {summary['mean_confidence']:.4f}")
    print(f"  confidence - accuracy : {summary['confidence_minus_accuracy']:+.4f}  "
          f"(positive = overconfident)")
    print(f"  ECE                   : {summary['ece']:.4f}")
    print(f"  Brier score           : {summary['brier_score']:.4f}")
    print(f"  NLL                   : {summary['nll']:.4f}")

    record_calibration_result({
        'model': model_name, 'stage': 'before_temperature', 'n_samples': summary['n_samples'],
        'accuracy': summary['accuracy'], 'mean_confidence': summary['mean_confidence'],
        'confidence_minus_accuracy': summary['confidence_minus_accuracy'],
        'ece': summary['ece'], 'brier_score': summary['brier_score'], 'nll': summary['nll'],
        'temperature': 1.0, 'notes': 'raw model output, no temperature scaling applied',
    })

--- MiniConvNet (test, before calibration) ---
  n_samples             : 315
  accuracy              : 0.4952
  mean_confidence       : 0.6609
  confidence - accuracy : +0.1657  (positive = overconfident)
  ECE                   : 0.1657
  Brier score           : 0.6570
  NLL                   : 1.2843
--- VGG16 (test, before calibration) ---
  n_samples             : 315
  accuracy              : 0.7714
  mean_confidence       : 0.7718
  confidence - accuracy : +0.0003  (positive = overconfident)
  ECE                   : 0.0265
  Brier score           : 0.3247
  NLL                   : 0.5801


## STEP 3 - reliability diagrams (before calibration)

In [7]:
reliability_paths = {}
for model_name in MODELS_AVAILABLE:
    test = predictions[(model_name, 'test')]
    p = plot_reliability_diagram(test['y_true'], test['y_prob'], model_name,
                                 method_label='raw', show=False)
    reliability_paths[(model_name, 'raw')] = p
    print(f'{model_name}: {p}')

MiniConvNet: C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\MiniConvNet_raw_reliability.png
VGG16: C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\VGG16_raw_reliability.png


## STEP 4 - temperature scaling, fit on VALIDATION ONLY

`fit_temperature()` only ever sees the validation-set arrays passed to it. Test-set probabilities are
touched for the first time in `before_after_comparison()`, which applies the already-fitted
temperature and reports the effect - the test set never participates in choosing T.

`apply_temperature()` works directly on the softmax outputs already available (see its docstring for
why this is an exact reimplementation of standard logit temperature scaling, not an approximation),
so no access to a pre-softmax layer inside the checkpoints is needed.

In [8]:
temperature_results = {}
for model_name in MODELS_AVAILABLE:
    val = predictions[(model_name, 'val')]
    test = predictions[(model_name, 'test')]

    fit = fit_temperature(val['y_true'], val['y_prob'])
    comparison = before_after_comparison(test['y_true'], test['y_prob'], fit['temperature'])
    temperature_results[model_name] = {'fit': fit, 'comparison': comparison}

    print(f'--- {model_name} ---')
    print(f"  fitted on {fit['val_n_samples']} VALIDATION samples only")
    print(f"  temperature           : {fit['temperature']:.4f}")
    print(f"  val NLL at T=1         : {fit['val_nll_at_T1']:.4f}")
    print(f"  val NLL at fitted T    : {fit['val_nll_at_fitted_T']:.4f}")
    print()
    print(f"  TEST accuracy before   : {comparison['before']['accuracy']:.4f}")
    print(f"  TEST accuracy after    : {comparison['after']['accuracy']:.4f}   "
          f"(accuracy_unchanged_as_expected={comparison['accuracy_unchanged_as_expected']})")
    print(f"  TEST ECE before        : {comparison['before']['ece']:.4f}")
    print(f"  TEST ECE after         : {comparison['after']['ece']:.4f}")
    print(f"  TEST Brier before      : {comparison['before']['brier_score']:.4f}")
    print(f"  TEST Brier after       : {comparison['after']['brier_score']:.4f}")
    print(f"  TEST mean_conf before  : {comparison['before']['mean_confidence']:.4f}")
    print(f"  TEST mean_conf after   : {comparison['after']['mean_confidence']:.4f}")
    print(f"  {comparison['note']}")

    reliability_paths[(model_name, 'temperature_scaled')] = plot_reliability_diagram(
        test['y_true'], comparison['y_prob_calibrated'], model_name,
        method_label='temperature_scaled', show=False)

    record_calibration_result({
        'model': model_name, 'stage': 'after_temperature',
        'n_samples': comparison['after']['n_samples'], 'accuracy': comparison['after']['accuracy'],
        'mean_confidence': comparison['after']['mean_confidence'],
        'confidence_minus_accuracy': comparison['after']['confidence_minus_accuracy'],
        'ece': comparison['after']['ece'], 'brier_score': comparison['after']['brier_score'],
        'nll': comparison['after']['nll'], 'temperature': fit['temperature'],
        'notes': f"T fitted on {fit['val_n_samples']} validation samples; applied to test only",
    })
    print()

--- MiniConvNet ---
  fitted on 72 VALIDATION samples only
  temperature           : 1.0056
  val NLL at T=1         : 0.6222
  val NLL at fitted T    : 0.6222

  TEST accuracy before   : 0.4952
  TEST accuracy after    : 0.4952   (accuracy_unchanged_as_expected=True)
  TEST ECE before        : 0.1657
  TEST ECE after         : 0.1644
  TEST Brier before      : 0.6570
  TEST Brier after       : 0.6563
  TEST mean_conf before  : 0.6609
  TEST mean_conf after   : 0.6597
  Temperature scaling is a monotonic per-sample rescaling of probabilities - it cannot and does not change any prediction's argmax, so accuracy before and after must be identical. This is a calibration-quality result only; it says nothing about, and must never be reported as, a classification-accuracy improvement.

--- VGG16 ---
  fitted on 72 VALIDATION samples only
  temperature           : 1.6520
  val NLL at T=1         : 0.7167
  val NLL at fitted T    : 0.6431

  TEST accuracy before   : 0.7714
  TEST accuracy after

## STEP 5 - MC Dropout (~15 stochastic forward passes, CPU-scoped, inference only)

`training=True` here reactivates Dropout's stochastic masking at call time - it does **not** trigger
any gradient computation or weight update; `mc_dropout_predict()` never calls `.fit()` or an
optimizer. `has_batchnorm()` guards against the one real correctness hazard (BatchNorm would also
react to `training=True`, contaminating the estimate) - both MiniConvNet (BatchNorm off by design)
and VGG16 (no BatchNorm layers exist in that architecture) are expected to pass this check, so MC
Dropout is attempted for **both** models here, not scoped to MiniConvNet alone - VGG16's single head
Dropout layer (rate 0.3, inherited from `models.build_baseline()`'s shared head) makes the identical
`training=True` mechanism equally applicable. Its estimate will be much less rich than
MiniConvNet's, though - VGG16's frozen/fine-tuned backbone contributes no stochasticity at all, only
its one head Dropout layer does, so expect a smaller effect there.

In [9]:
N_MC_PASSES = 15   # within the requested 10-20 range

mc_results = {}
mc_raw_results = {}   # model_name -> raw mc_dropout_predict() output, for Step 6 plots
for model_name in MODELS_AVAILABLE:
    test = predictions[(model_name, 'test')]
    model = test.get('model')
    if model is None:
        # predictions were loaded from a saved file rather than generated this run,
        # so no in-memory model object exists yet - load the checkpoint fresh for MC
        # Dropout (still inference only, never training).
        import tensorflow as tf
        ck = PRIMARY_CHECKPOINT if model_name == 'MiniConvNet' else SECONDARY_CHECKPOINT
        print(f'{model_name}: loading checkpoint for MC Dropout (inference only): {ck}')
        model = tf.keras.models.load_model(ck)

    if has_batchnorm(model):
        print(f'{model_name}: SKIPPED - contains BatchNormalization layer(s); training=True would '
              'also perturb BN statistics, contaminating the uncertainty estimate (see Step 5 note).')
        continue

    from src.data_utils import make_dataset
    frame = sdf[sdf['split'] == 'test'].reset_index(drop=True)
    one_hot = (model_name == 'MiniConvNet')
    ds = make_dataset(frame, one_hot=one_hot)

    print(f'--- {model_name}: MC Dropout, {N_MC_PASSES} passes over {len(frame)} test images ---')
    t0 = time.time()
    result = mc_dropout_predict(model, ds, n_passes=1, verbose=False)   # time one pass first
    one_pass_seconds = time.time() - t0
    est = print_mc_dropout_time_estimate(one_pass_seconds, N_MC_PASSES, label=model_name)

    t0 = time.time()
    result = mc_dropout_predict(model, ds, n_passes=N_MC_PASSES, verbose=True)
    total_seconds = time.time() - t0
    print(f'  actual total time: {total_seconds:.1f}s')

    cvi = mc_dropout_correct_vs_incorrect(result)
    print(f"  correct   : n={cvi['correct']['n']:4d}  mean_entropy={cvi['correct'].get('mean_entropy', float('nan')):.4f}")
    print(f"  incorrect : n={cvi['incorrect']['n']:4d}  mean_entropy={cvi['incorrect'].get('mean_entropy', float('nan')):.4f}")

    mc_results[model_name] = {
        'n_passes': result['n_passes'], 'n_samples': result['n_samples'],
        'mean_accuracy': float(np.mean(result['y_pred_mean'] == result['y_true'])),
        'mean_predictive_entropy': float(result['predictive_entropy'].mean()),
        'mean_confidence_variance': float(result['confidence_variance'].mean()),
        'correct_vs_incorrect': cvi, 'total_seconds': round(total_seconds, 1),
    }
    mc_raw_results[model_name] = result   # keep raw arrays for the Step 6 plot

if mc_results:
    save_mc_dropout_summary(mc_results)
    print()
    print('MC Dropout summary saved to', MC_DROPOUT_RESULTS_JSON)
else:
    print('No model was eligible for MC Dropout this run.')

--- MiniConvNet: MC Dropout, 15 passes over 315 test images ---
[MiniConvNet] first pass took 0.7s -> ~9.8s (0.2 min) for all 15 passes
    MC Dropout pass 1/15 done (315 samples)
    MC Dropout pass 2/15 done (315 samples)
    MC Dropout pass 3/15 done (315 samples)
    MC Dropout pass 4/15 done (315 samples)
    MC Dropout pass 5/15 done (315 samples)
    MC Dropout pass 6/15 done (315 samples)
    MC Dropout pass 7/15 done (315 samples)
    MC Dropout pass 8/15 done (315 samples)
    MC Dropout pass 9/15 done (315 samples)
    MC Dropout pass 10/15 done (315 samples)
    MC Dropout pass 11/15 done (315 samples)
    MC Dropout pass 12/15 done (315 samples)
    MC Dropout pass 13/15 done (315 samples)
    MC Dropout pass 14/15 done (315 samples)
    MC Dropout pass 15/15 done (315 samples)
  actual total time: 9.0s
  correct   : n= 152  mean_entropy=0.6942
  incorrect : n= 163  mean_entropy=1.0689
VGG16: loading checkpoint for MC Dropout (inference only): C:\Users\shrey\Downloads\MIP_

## STEP 6 - visualisations

Confidence distribution, correct-vs-incorrect confidence, and (for models MC Dropout ran on)
uncertainty-vs-correctness, for every available model.

In [10]:
figure_paths = []
for model_name in MODELS_AVAILABLE:
    test = predictions[(model_name, 'test')]
    figure_paths.append(plot_confidence_distribution(test['y_true'], test['y_prob'], model_name,
                                                      method_label='raw', show=False))
    figure_paths.append(plot_confidence_by_correctness(test['y_true'], test['y_prob'], model_name,
                                                        method_label='raw', show=False))
    mc_var = mc_raw_results.get(model_name)
    if mc_var is not None:
        figure_paths.append(plot_uncertainty_vs_correctness(mc_var, model_name, show=False))

print(f'{len([p for p in figure_paths if p])} figures written to {CALIBRATION_FIGURES_DIR}')
for p in figure_paths:
    if p:
        print(' ', p)

6 figures written to C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\MiniConvNet_raw_confidence_dist.png
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\MiniConvNet_raw_confidence_by_correctness.png
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\MiniConvNet_mc_dropout_uncertainty_vs_correctness.png
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\VGG16_raw_confidence_dist.png
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration\figures\VGG16_raw_confidence_by_correctness.png
 

## STEP 7 - interpretation

Fill in once the cells above have actually been run. **Calibration quality and classification
performance are kept explicitly separate below - do not conflate them.** Temperature scaling is a
monotonic rescaling of probabilities; it mathematically cannot change accuracy (asserted by
`accuracy_unchanged_as_expected` above), so any accuracy claim about it would be wrong by
construction.

**1. Is the model overconfident?**
*(look at `confidence_minus_accuracy` from Step 2 for each model - positive means overconfident,*
*negative means underconfident)*

**2. What is ECE before calibration?**
*(from Step 2's printed values / `results/calibration/calibration_metrics.csv`, stage='before_temperature')*

**3. What is ECE after calibration?**
*(from Step 4's printed values / the same CSV, stage='after_temperature')*

**4. Does temperature scaling improve calibration?**
*(compare ECE and Brier before vs after from Step 4 - state plainly if it does not for either model;*
*do NOT claim it changes accuracy - it cannot, by construction)*

**5. Are uncertain predictions more likely to be incorrect?**
*(from Step 5's correct-vs-incorrect mean entropy comparison, and the Step 6 boxplot - higher entropy*
*among incorrect predictions would support this; report what was actually found)*

**6. Does MC Dropout provide useful uncertainty?**
*(state which models it ran on - and if either was skipped, exactly why, per the BatchNorm guard);*
*note VGG16's MC-Dropout signal is expected to be weaker than MiniConvNet's, since only one head*
*Dropout layer contributes to it there versus MiniConvNet's dropout sitting after its trained*
*bottleneck*

In [11]:
calib_df = load_calibration_results()

print('PHASE:                Phase 4 - calibration and uncertainty')
print('MODELS:                MiniConvNet (primary)'
      + (f', {SECONDARY_MODEL_NAME} (secondary, strongest Phase 2 fine-tuned baseline)'
         if SECONDARY_MODEL_NAME else ' | secondary: NOT DETERMINED'))
print()

for model_name in MODELS_AVAILABLE:
    b = calib_df[(calib_df['model'] == model_name) & (calib_df['stage'] == 'before_temperature')]
    a = calib_df[(calib_df['model'] == model_name) & (calib_df['stage'] == 'after_temperature')]
    print(f'--- {model_name} ---')
    print(f"ECE BEFORE:            {b['ece'].iloc[0]:.4f}" if len(b) else 'ECE BEFORE:            n/a')
    print(f"ECE AFTER:             {a['ece'].iloc[0]:.4f}" if len(a) else 'ECE AFTER:             n/a')
    print(f"BRIER BEFORE:          {b['brier_score'].iloc[0]:.4f}" if len(b) else 'BRIER BEFORE:          n/a')
    print(f"BRIER AFTER:           {a['brier_score'].iloc[0]:.4f}" if len(a) else 'BRIER AFTER:           n/a')
    print(f"TEMPERATURE:           {a['temperature'].iloc[0]:.4f}" if len(a) else 'TEMPERATURE:           n/a')
    mc = mc_results.get(model_name)
    if mc:
        print(f"MC DROPOUT RESULT:     {mc['n_passes']} passes; mean entropy "
              f"correct={mc['correct_vs_incorrect']['correct'].get('mean_entropy', float('nan')):.4f} "
              f"vs incorrect={mc['correct_vs_incorrect']['incorrect'].get('mean_entropy', float('nan')):.4f}")
    else:
        print('MC DROPOUT RESULT:     skipped for this model (see Step 5 - BatchNorm guard or not run)')
    print()

print('FILES CREATED:')
print(f'  {CALIBRATION_DIR}/calibration_metrics.csv')
print(f'  {CALIBRATION_DIR}/calibration_metrics.json')
print(f'  {CALIBRATION_DIR}/mc_dropout_results.json')
print(f'  {CALIBRATION_DIR}/predictions_used/  (mirrored copies with provenance, per model/split)')
print(f'  {CALIBRATION_FIGURES_DIR}/  ({len([p for p in figure_paths if p]) + len(reliability_paths)} figures)')
print('CPU TIME:              sum of the per-model timings printed above (MC Dropout is the bulk of it)')
print('INTERPRETATION:        fill in Step 7 above once run')
print('LIMITATIONS:           temperature scaling assumes a SINGLE global scalar is enough (it cannot')
print('                       fix per-class miscalibration); ECE is bin-count-sensitive (n_bins=10,')
print('                       not swept); MC Dropout uses the checkpoint\'s already-fixed dropout rate')
print('                       (not re-tuned for uncertainty quality); VGG16\'s MC-Dropout signal, if')
print('                       run, reflects only its single head Dropout layer, not its backbone;')
print('                       validation set is small (faithful split, ~70 images), so the fitted')
print('                       temperature itself carries meaningful uncertainty not quantified here.')

PHASE:                Phase 4 - calibration and uncertainty
MODELS:                MiniConvNet (primary), VGG16 (secondary, strongest Phase 2 fine-tuned baseline)

--- MiniConvNet ---
ECE BEFORE:            0.1657
ECE AFTER:             0.1644
BRIER BEFORE:          0.6570
BRIER AFTER:           0.6563
TEMPERATURE:           1.0056
MC DROPOUT RESULT:     15 passes; mean entropy correct=0.6942 vs incorrect=1.0689

--- VGG16 ---
ECE BEFORE:            0.0265
ECE AFTER:             0.1207
BRIER BEFORE:          0.3247
BRIER AFTER:           0.3449
TEMPERATURE:           1.6520
MC DROPOUT RESULT:     15 passes; mean entropy correct=0.6416 vs incorrect=0.9438

FILES CREATED:
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration/calibration_metrics.csv
  C:\Users\shrey\Downloads\MIP_project_part2\Lung-Cancer-Classification-\Lung-Cancer-Classification-\results\calibration/calibration_metrics.json
  C:\Users\shrey\Downloads\MIP